# Phase 6 — counting passage TOR3-PAGI dengan Opsi A

Notebook CPU ini memproses **diagnostik crossing yang sudah tersimpan** menjadi
keputusan dan event yang dapat diaudit. Ia tidak menjalankan ulang YOLO,
BoT-SORT, ReID, geometri, atau decode video.

**USER DECISION:** Opsi A: accepted hanya jika bukti geometri, unit, kategori
dan identitas memenuhi aturan eksplisit; konflik ditahan. **ENGINEERING
EXPERIMENT:** baseline awal audit-gated. Setiap kandidat tanpa review positif
tetap pending, termasuk kandidat tanpa konflik otomatis yang ditemukan.

Kontrak: [COUNTING_CONTRACT.md](../docs/COUNTING_CONTRACT.md).
Kontrol geometri tersimpan `b0_c1` memakai band ±0 native px dan satu
konfirmasi. Angka ini bukan parameter produk terbaik atau pilihan numerik
pengguna. Metrik prototype adalah passage, bukan kendaraan unik.

**Batas:** accepted subtotal bukan total traffic lengkap yang tervalidasi.
Zero accepted tanpa review tidak berarti tidak ada kendaraan. Belum ada
ground truth; tidak menghasilkan accuracy/precision/recall/F1/MAPE.

## Cara menjalankan dan mengganti input

Di Codex, buka notebook dari root checkout atau direktori `notebooks/`, lalu
jalankan seluruh sel dengan Python. Helper dan tes memakai standard library;
IPython hanya opsional untuk menampilkan gambar.

Di Kaggle, sediakan checkout GitHub lengkap yang berisi helper, tes,
diagnostik Phase 5, cache Phase 4, geometri, konfigurasi serta evidence review.
Tetapkan `REPO_ROOT` ke lokasi checkout aktual dan sesuaikan path di bawah.
GPU dan file video sumber tidak diperlukan untuk core counting.

Untuk mesin saja, tetapkan `REVIEW_PATH = None` secara eksplisit. Untuk
review yang direproduksi, gunakan JSON dengan binding input dan evidence
pixel ber-hash sesuai kontrak. Path review/evidence yang hilang harus
disediakan; notebook tidak menggantinya dengan hasil review palsu. Mengganti
video memerlukan review geometri, input identity dan konfigurasi baru.

Restart kernel lalu Run All direkomendasikan. Sel import juga reload helper
sebelum tes sehingga rerun memakai kode checkout terkini. Event IDs tidak
bergantung pada waktu run; setiap eksekusi menulis snapshot terpisah.

In [ ]:
from pathlib import Path

REPO_ROOT = Path.cwd().resolve()
if not (REPO_ROOT / 'AGENTS.md').is_file() and (REPO_ROOT.parent / 'AGENTS.md').is_file():
    REPO_ROOT = REPO_ROOT.parent

DIAGNOSTICS_PATH = REPO_ROOT / (
    'docs/evidence/phase_5/20261011T161440_100155Z_2e39a11c/'
    'b0_c1_diagnostics.jsonl.gz'
)
RECORDS_PATH = REPO_ROOT / (
    'docs/evidence/phase_4/20261011T095924_205716Z_90cf7319/'
    'reid_on/track_records.jsonl.gz'
)
GEOMETRY_PATH = REPO_ROOT / 'docs/evidence/phase_5/accepted_geometry/TOR3-PAGI-G3.json'
CONFIG_PATH = REPO_ROOT / 'docs/evidence/phase_6/config/TOR3-PAGI_option_a_v0.json'
REVIEW_PATH = REPO_ROOT / 'docs/evidence/phase_6/reviews/TOR3-PAGI_option_a_reviews.json'
# REVIEW_PATH = None  # Mesin saja: semua belum direview tetap pending.
EVIDENCE_ROOT = REPO_ROOT
OUTPUT_DIR = (
    Path('/kaggle/working/phase_6_counting')
    if Path('/kaggle/working').is_dir()
    else Path('/tmp/phase_6_counting')
)
REVIEW_SAMPLE_INDEX_PATH = (
    REPO_ROOT / 'docs/evidence/phase_6/review_samples/TOR3-PAGI/sample_index.json'
)
# Opsional: hanya tampilkan contact sheets, tidak menjalankan review otomatis.
DISPLAY_REVIEW_TRACK_IDS = (1, 973, 1774)

## Lingkungan dan source code

Pemeriksaan di sini hanya memeriksa keberadaan path dan mengimpor kode.
Isi diagnostik/cache/config/review aktual baru dibaca setelah guard tes.
Tidak memeriksa atau mencetak token, credential, maupun environment secrets.

In [ ]:
import sys
import os
import importlib
import json
import gzip
import hashlib
import platform
import time
import csv
from datetime import datetime, timezone
from collections import Counter

REPO_ROOT = Path(REPO_ROOT).resolve()
DIAGNOSTICS_PATH = Path(DIAGNOSTICS_PATH).resolve()
RECORDS_PATH = Path(RECORDS_PATH).resolve()
GEOMETRY_PATH = Path(GEOMETRY_PATH).resolve()
CONFIG_PATH = Path(CONFIG_PATH).resolve()
REVIEW_PATH = None if REVIEW_PATH is None else Path(REVIEW_PATH).resolve()
EVIDENCE_ROOT = Path(EVIDENCE_ROOT).resolve()
OUTPUT_DIR = Path(OUTPUT_DIR).resolve()
HELPER_DIR = REPO_ROOT / 'notebooks'

required = {
    'geometry validation helper': HELPER_DIR / 'phase_5_geometry_helpers.py',
    'counting helper': HELPER_DIR / 'phase_6_counting_helpers.py',
    'synthetic tests': HELPER_DIR / 'phase_6_synthetic_tests.py',
    'retained geometry diagnostics': DIAGNOSTICS_PATH,
    'retained track records': RECORDS_PATH,
    'accepted geometry': GEOMETRY_PATH,
    'counting configuration': CONFIG_PATH,
}
if REVIEW_PATH is not None:
    required['explicit review input'] = REVIEW_PATH
missing = {name: str(path) for name, path in required.items() if not path.is_file()}
if missing:
    raise FileNotFoundError(
        'Sediakan checkout/input aktual atau perbaiki parameter path. '
        'Untuk mesin saja pilih REVIEW_PATH=None secara eksplisit:\n'
        + json.dumps(missing, indent=2)
    )
if not EVIDENCE_ROOT.is_dir():
    raise FileNotFoundError('EVIDENCE_ROOT belum tersedia: ' + str(EVIDENCE_ROOT))

sys.path.insert(0, str(HELPER_DIR))
importlib.invalidate_caches()
import phase_5_geometry_helpers as geometry_helpers
geometry_helpers = importlib.reload(geometry_helpers)
import phase_6_counting_helpers as counting_helpers
import phase_6_synthetic_tests as synthetic_tests
counting_helpers = importlib.reload(counting_helpers)
synthetic_tests = importlib.reload(synthetic_tests)
for module in (geometry_helpers, counting_helpers, synthetic_tests):
    if Path(module.__file__).resolve().parent != HELPER_DIR.resolve():
        raise RuntimeError('Import berasal dari checkout lain; restart kernel lalu Run All.')

def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as source:
        for block in iter(lambda: source.read(1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()

def write_json(path, payload):
    Path(path).write_text(
        json.dumps(payload, ensure_ascii=False, allow_nan=False, sort_keys=True, indent=2) + '\n',
        encoding='utf-8',
    )

def write_jsonl_gzip(path, rows):
    with Path(path).open('wb') as output:
        with gzip.GzipFile(filename='', mode='wb', fileobj=output, mtime=0) as compressed:
            for row in rows:
                compressed.write((json.dumps(
                    row, ensure_ascii=False, allow_nan=False, sort_keys=True
                ) + '\n').encode('utf-8'))

runtime = {
    'python': sys.version,
    'python_executable': sys.executable,
    'platform': platform.platform(),
    'available_cpu_count': os.cpu_count(),
    'execution_mode': 'CPU_ONLY_RETAINED_CROSSING_COUNTING_AUDIT',
    'gpu_required': False,
    'video_decode_required_for_counting': False,
    'models_and_geometry_rerun': False,
}
print(json.dumps(runtime, indent=2))

## Guard: tes sintetis counting sebelum input aktual

Fixture tes adalah bukti correctness software. Ia bukan observasi video nyata
atau ground truth. Jika gagal, hasil tes disimpan lalu counting dihentikan.

In [ ]:
run_started_at = datetime.now(timezone.utc).isoformat()
# Timestamp hanya nama direktori/log eksekusi, tidak masuk logical event ID.
run_id = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S_%fZ')
RUN_DIR = OUTPUT_DIR / run_id
RUN_DIR.mkdir(parents=True, exist_ok=False)
source_code_sha256 = {
    'geometry_validation_helper': sha256_file(geometry_helpers.__file__),
    'counting_helper': sha256_file(counting_helpers.__file__),
    'synthetic_tests': sha256_file(synthetic_tests.__file__),
}
notebook_source = HELPER_DIR / '06_counting_audit.ipynb'
if notebook_source.is_file():
    source_code_sha256['repository_notebook_source'] = sha256_file(notebook_source)

test_result = synthetic_tests.run_tests()
write_json(RUN_DIR / 'synthetic_test_results.json', test_result)
print(json.dumps({key: test_result[key] for key in (
    'successful', 'tests_run', 'tests_passed', 'failures', 'errors', 'skipped'
)}, indent=2))
if not test_result['successful']:
    raise AssertionError('Tes gagal; counting tidak dijalankan. Bukti: ' + str(RUN_DIR))
print('Guard lulus; output snapshot:', RUN_DIR)

## Binding input dan konfigurasi eksperimen

Hash aktual dibandingkan dengan `input_identity` konfigurasi, bukan dianggap
benar dari filename. Lebar/tinggi, variant dan source links juga diverifikasi
helper. Metadata yang tidak diketahui tetap null. Input review diretain
terpisah dari keputusan mesin; tidak mengedit subtotal secara langsung.

In [ ]:
config = json.loads(CONFIG_PATH.read_text(encoding='utf-8'))
reviews = None if REVIEW_PATH is None else json.loads(REVIEW_PATH.read_text(encoding='utf-8'))
input_sha256 = {
    'diagnostics': sha256_file(DIAGNOSTICS_PATH),
    'records': sha256_file(RECORDS_PATH),
    'geometry': sha256_file(GEOMETRY_PATH),
    'configuration': sha256_file(CONFIG_PATH),
    'reviews': None if REVIEW_PATH is None else sha256_file(REVIEW_PATH),
}
expected = config['input_identity']
for name in ('diagnostics', 'records', 'geometry'):
    if input_sha256[name] != expected[name + '_sha256']:
        raise ValueError('Hash input berbeda dari konfigurasi: ' + name)
if config.get('metric') != 'passage' or config.get('review_policy') != 'audit_gated':
    raise ValueError('Notebook ini khusus kontrak passage audit_gated Opsi A.')
if not config.get('parameter_claim_type'):
    raise ValueError('Basis parameter eksperimen harus eksplisit.')

# Copy bytes persis agar provenance JSON tidak bergantung reformat notebook.
(RUN_DIR / 'configuration_input.json').write_bytes(CONFIG_PATH.read_bytes())
if REVIEW_PATH is not None:
    (RUN_DIR / 'review_input.json').write_bytes(REVIEW_PATH.read_bytes())
else:
    write_json(RUN_DIR / 'review_input.json', None)
write_json(RUN_DIR / 'runtime.json', runtime)
print(json.dumps({
    'contract_version': config['contract_version'],
    'metric': config['metric'],
    'review_policy': config['review_policy'],
    'neutral_band_px_halfwidth': config['neutral_band_px'],
    'confirmation_observations': config['confirmation_observations'],
    'parameter_claim_type': config['parameter_claim_type'],
    'review_input_supplied': reviews is not None,
    'input_sha256': input_sha256,
}, indent=2))

## Counting dan resolusi audit

Hanya `GEOMETRY_TRANSITION` terukur menjadi kandidat. Seluruh histori kelas
origin–confirmation diperiksa. Unit-changing history, gap/endpoint/prediction
crossing dan taxonomy yang belum ditetapkan tidak dapat diterima lewat
override review. Same physical unit + same passage menghasilkan satu event;
genuine return memerlukan bukti episode tersendiri.

Machine outputs dan resolved outputs berikut disimpan terpisah. Pada baseline
audit-gated, mesin tanpa review tidak menghasilkan accepted events.

In [ ]:
counting_started_at = datetime.now(timezone.utc).isoformat()
counting_clock = time.perf_counter()
result = counting_helpers.run_counting(
    DIAGNOSTICS_PATH, RECORDS_PATH, GEOMETRY_PATH,
    config=config, reviews=reviews, evidence_root=EVIDENCE_ROOT,
)
elapsed_seconds = time.perf_counter() - counting_clock
required_result_keys = {
    'machine_decisions', 'resolved_decisions', 'machine_events',
    'accepted_events', 'machine_aggregate', 'aggregate', 'report',
}
if not required_result_keys.issubset(result):
    raise ValueError('Output helper tidak memenuhi interface kontrak.')

accepted_events = result['accepted_events']
accepted_ids = [event['event_id'] for event in accepted_events]
if len(accepted_ids) != len(set(accepted_ids)):
    raise AssertionError('Duplicate accepted event_id.')
aggregate = result['aggregate']
category_counts = Counter(event['category'] for event in accepted_events)
direction_counts = Counter(event['direction'] for event in accepted_events)
if aggregate['total'] != len(accepted_events):
    raise AssertionError('Aggregate total tidak sama dengan accepted event unik.')
for field, actual_counts in (('by_category', category_counts), ('by_direction', direction_counts)):
    if set(actual_counts) - set(aggregate[field]):
        raise AssertionError('Aggregate kehilangan key: ' + field)
    if any(aggregate[field][key] != actual_counts.get(key, 0) for key in aggregate[field]):
        raise AssertionError('Aggregate tidak reconcile: ' + field)
if sorted(aggregate['event_ids']) != sorted(accepted_ids):
    raise AssertionError('Aggregate event references berbeda dari accepted output.')
if result['machine_events'] or result['machine_aggregate']['total'] != 0:
    raise AssertionError('Audit-gated v0 machine subtotal harus nol sebelum review positif.')

for name in ('machine_decisions', 'resolved_decisions', 'machine_events', 'accepted_events'):
    write_jsonl_gzip(RUN_DIR / (name + '.jsonl.gz'), result[name])
for name in ('machine_aggregate', 'aggregate'):
    write_json(RUN_DIR / (name + '.json'), result[name])
execution_report = {
    'run_id': run_id,
    'claim_type': 'EXPERIMENT_RESULT_AUDIT_GATED_PASSAGE_SUBTOTAL_NOT_COMPLETE_TRAFFIC',
    'counting_started_at_utc': counting_started_at,
    'counting_completed_at_utc': datetime.now(timezone.utc).isoformat(),
    'measured_counting_elapsed_seconds': elapsed_seconds,
    'runtime': runtime,
    'source_code_sha256': source_code_sha256,
    'input_sha256': input_sha256,
    'effective_configuration': config,
    'review_input_supplied': reviews is not None,
    'synthetic_tests_successful': test_result['successful'],
    'synthetic_test_results_sha256': sha256_file(RUN_DIR / 'synthetic_test_results.json'),
    'software_report': result['report'],
    'selected_suitable_product_parameters': None,
    'source_pts_status': 'UNKNOWN_FRAME_FPS_TIMESTAMPS_ARE_ESTIMATES',
    'whole_video_physical_audit_complete': False,
    'formal_ground_truth_available': False,
    'phase_7_started': False,
}
write_json(RUN_DIR / 'report.json', execution_report)
print('Counting CPU selesai:', round(elapsed_seconds, 3), 'detik')

## Subtotal accepted dan audit queue

Jumlah accepted event unik berbeda dari jumlah candidate decisions accepted:
beberapa sumber duplicate dapat merujuk satu event. Pending dan rejected tidak
masuk subtotal. CSV berikut hanya daftar pekerjaan review, bukan override yang
dapat diterima helper. Gunakan schema review JSON kontrak untuk penyelesaian.

In [ ]:
status_counts = dict(Counter(row['status'] for row in result['resolved_decisions']))
pending = [row for row in result['resolved_decisions'] if row['status'] == 'pending_review']
rejected = [row for row in result['resolved_decisions'] if row['status'] == 'rejected']
write_jsonl_gzip(RUN_DIR / 'audit_queue.jsonl.gz', pending)
write_jsonl_gzip(RUN_DIR / 'rejected_decisions.jsonl.gz', rejected)

def reason_codes(row):
    reasons = row.get('reasons', row.get('reason_codes', []))
    if isinstance(reasons, str):
        reasons = [reasons]
    return [reason if isinstance(reason, str) else str(reason) for reason in reasons]

pending_reason_counts = dict(Counter(reason for row in pending for reason in reason_codes(row)))
rejected_reason_counts = dict(Counter(reason for row in rejected for reason in reason_codes(row)))
summary = {
    'resolved_candidate_status_counts': status_counts,
    'accepted_unique_event_subtotal': len(result['accepted_events']),
    'pending_reason_frequencies_nonexclusive': pending_reason_counts,
    'rejected_reason_frequencies_nonexclusive': rejected_reason_counts,
    'aggregate_from_accepted_events': result['aggregate'],
    'machine_aggregate_separate': result['machine_aggregate'],
    'interpretation': 'ACCEPTED_SUBTOTAL_NOT_COMPLETE_TRAFFIC_COUNT',
}
write_json(RUN_DIR / 'summary.json', summary)
print(json.dumps(summary, indent=2, ensure_ascii=False))

with (RUN_DIR / 'review_queue.csv').open('w', encoding='utf-8', newline='') as stream:
    writer = csv.DictWriter(stream, fieldnames=(
        'candidate_id', 'status', 'reasons', 'candidate_evidence_json', 'required_input_binding_json',
    ))
    writer.writeheader()
    for row in pending:
        writer.writerow({
            'candidate_id': row['candidate_id'],
            'status': row['status'],
            'reasons': '|'.join(reason_codes(row)),
            'candidate_evidence_json': json.dumps(row, ensure_ascii=False, sort_keys=True),
            'required_input_binding_json': json.dumps(expected, sort_keys=True),
        })

## Validasi snapshot dan manifest

Input/source hash diperiksa kembali untuk mendeteksi perubahan saat run.
Manifest merekam file output beserta hash, bukan membuktikan interpretasi
visual benar. Rerun menghasilkan snapshot baru; total antar snapshot tidak
ditambahkan. Pemeriksaan idempotensi/reconciliation juga ada dalam tes.

In [ ]:
for name, path in (
    ('diagnostics', DIAGNOSTICS_PATH), ('records', RECORDS_PATH),
    ('geometry', GEOMETRY_PATH), ('configuration', CONFIG_PATH),
):
    if sha256_file(path) != input_sha256[name]:
        raise RuntimeError('Input berubah selama run: ' + name)
if REVIEW_PATH is not None and sha256_file(REVIEW_PATH) != input_sha256['reviews']:
    raise RuntimeError('Review input berubah selama run.')
for name, path in (
    ('geometry_validation_helper', geometry_helpers.__file__),
    ('counting_helper', counting_helpers.__file__), ('synthetic_tests', synthetic_tests.__file__),
):
    if sha256_file(path) != source_code_sha256[name]:
        raise RuntimeError('Source code berubah selama run: ' + name)
event_ids = [event['event_id'] for event in result['accepted_events']]
if len(event_ids) != len(set(event_ids)):
    raise AssertionError('Snapshot memiliki duplicate accepted event_id.')
if reviews is None and result['accepted_events']:
    raise AssertionError('Audit-gated tanpa review tidak boleh memiliki accepted events.')
if result['machine_events']:
    raise AssertionError('Audit-gated v0 tidak menerima event mesin tanpa review positif.')

artifact_paths = sorted(path for path in RUN_DIR.iterdir() if path.is_file())
manifest = {
    'run_id': run_id,
    'run_started_at_utc': run_started_at,
    'run_completed_at_utc': datetime.now(timezone.utc).isoformat(),
    'claim_type': 'EXPERIMENT_RESULT_PHASE6_AUDIT_GATED_SOFTWARE',
    'source_code_sha256': source_code_sha256,
    'input_sha256': input_sha256,
    'accepted_unique_event_subtotal': len(result['accepted_events']),
    'artifacts': [
        {'path': path.name, 'size_bytes': path.stat().st_size, 'sha256': sha256_file(path)}
        for path in artifact_paths
    ],
}
write_json(RUN_DIR / 'execution_manifest.json', manifest)
print('Snapshot dan manifest:', RUN_DIR)
print('Langkah berikutnya: review pending dengan evidence terukur; jangan edit aggregate langsung.')
print('Phase 7 atau fase selanjutnya belum dimulai.')

## Opsional: tampilkan crossing contact sheets aktual

Contact sheets ini mempertahankan frame indices, crop mapping dan measurement
links. Overlay adalah visualisasi audit, bukan hasil model baru. Sel ini hanya
menampilkan artefak yang hash-nya sesuai index; ia tidak membuat review
positif atau memvalidasi seluruh video. Raw sheet dan overlay perlu dibaca
bersama untuk membedakan kendaraan, rider dan kandidat ID yang bertumpuk.

In [ ]:
sample_index_path = Path(REVIEW_SAMPLE_INDEX_PATH).resolve()
if not sample_index_path.is_file():
    print('Contact-sheet index tidak tersedia; tidak ada review visual yang dijalankan di sel ini.')
else:
    sample_index = json.loads(sample_index_path.read_text(encoding='utf-8'))
    if sample_index.get('source_video_sha256') != expected['video_sha256']:
        raise ValueError('Contact-sheet index berasal dari video berbeda.')
    try:
        from IPython.display import Image, display
    except ImportError:
        Image = display = None
    for case in sample_index['cases']:
        if case['track_id'] not in DISPLAY_REVIEW_TRACK_IDS:
            continue
        print('Track', case['track_id'], 'frame tersedia:', case['frames'])
        for kind in ('raw', 'overlay'):
            artifact = case['artifacts'][kind]
            image_path = EVIDENCE_ROOT / artifact['path']
            if not image_path.is_file() or sha256_file(image_path) != artifact['sha256']:
                raise ValueError('Contact sheet hilang/berubah: ' + str(image_path))
            if display is None:
                print(kind, image_path)
            else:
                print(kind)
                display(Image(filename=str(image_path)))